# Deploy AnthroFold to SageMaker

This notebook stands up an AnthroFold inference endpoint on Amazon SageMaker. AnthroFold is a structure-prediction model specialised for antibody-antigen complexes — you give it sequences and it returns predicted 3D structures as mmCIF along with per-prediction confidence scores.

It is the first of five notebooks. The full workflow is:

1. **Deploy** (this notebook) — stand up the endpoint.
2. **Invoke** (`2-invoke-endpoint.ipynb`) — submit your input CSV, save the returned structures.
3. **Score with DockQ** (`4-score-dockq.ipynb`, optional) — quantify how close predicted structures are to your experimental ground truth.
4. **Predict the epitope** (`5-predict-epitope.ipynb`, optional) — extract the antigen residues each prediction places at the interface.
5. **Cleanup** (`3-cleanup-endpoint.ipynb`) — tear the endpoint down when you're finished, so it stops billing.

Notebooks 4 and 5 are independent and can be run in either order, or skipped.

**A few practical notes before you start:**

- **Cold start takes about an hour.** The container syncs a bundled set of MSA databases before serving traffic. After that, individual predictions take roughly 5-10 minutes each, depending on complex size.
- **Use an 80 GB-class GPU** such as `ml.p4de.24xlarge` or `ml.p5.48xlarge`. The model supports protein complexes up to **2048 residues total**, and 80 GB of GPU memory comfortably covers that range. Smaller shapes (e.g. `ml.g5.12xlarge`, 4× A10G 24 GB) can serve smaller complexes if those instance types aren't available.
- **If you hit `InsufficientInstanceCapacity`,** the deploy cell cleans up after itself — just re-run it. Capacity for these instance types fluctuates throughout the day.
- **Training and template cutoff is 2021-09-30,** matching the AlphaFold 3 protocol. PDB structures released after that date are filtered out of the template search at inference time.

If anything goes wrong mid-deploy, `3-cleanup-endpoint.ipynb` can tear everything down cleanly.

## Input format

`2-invoke-endpoint.ipynb` reads a six-column CSV that describes each prediction job. Prepare your inputs in this format before running the inference notebook.

| Column          | Required | Description |
| --------------- | -------- | ----------- |
| `antigen_seq`   | yes | Antigen amino-acid sequence. Multiple antigen chains may be separated with `/`. |
| `binder_seq`    | yes | Binder amino-acid sequence. Multiple binder chains separated with `/`. |
| `binder_mode`   | yes | `1` for antibody-like binders, `0` for general protein binders. |
| `antibody_form` | yes | `VH/VL`, `SS`, or `None`. |
| `antigen_temp`  | no  | Antigen template field from the input schema, or `None`. |
| `exp_structure` | recommended | Experimental ground-truth structure filename. Required for the bundled DockQ workflow so predictions can be matched back to their source ground truth. |

See `examples/structure_determination_input.csv` for four real antibody-antigen examples. Drop your own CSV at any path and set `INPUT_PATH` in notebook 2.

## Install dependencies

In [ ]:
%pip install -qU -r requirements.txt

## Configure

Set the ModelPackage ARN and pick an instance type. In SageMaker Studio your execution role is detected automatically; for local runs, uncomment the credentials block and point boto3 at your AWS profile.

In [ ]:
import getpass
import os
import time

import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.async_inference import AsyncInferenceConfig
from sagemaker.model import ModelPackage

SAGEMAKER_REGION = "us-east-1"
S3_BUCKET_REGION = "us-east-1"
# In SageMaker Studio, credentials come from the attached execution role; leave the
# block below commented. If running locally, point boto3 at your AWS credentials file.
# AWS_CREDS_PATH = "/path/to/.aws/credentials"
# os.environ["AWS_SHARED_CREDENTIALS_FILE"] = AWS_CREDS_PATH
# os.environ["AWS_PROFILE"] = "default"
os.environ["AWS_DEFAULT_REGION"] = SAGEMAKER_REGION
os.environ["AWS_REGION"] = SAGEMAKER_REGION

MODEL_PACKAGE_ARN = "arn:aws:sagemaker:us-east-1:038462780959:model-package/anthrofold-v1/4"
EXECUTION_ROLE_ARN = None  # leave None to use SageMaker Studio's attached role
INSTANCE_TYPE = "ml.p4de.24xlarge"  # 8 x A100 80GB; recommended for the full Ab-Ag size range

boto_session = boto3.Session(region_name=SAGEMAKER_REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session)
sm = boto_session.client("sagemaker")
s3 = boto3.client("s3", region_name=S3_BUCKET_REGION)
logs = boto_session.client("logs")
account_id = boto_session.client("sts").get_caller_identity()["Account"]

role = EXECUTION_ROLE_ARN
if role is None:
    role = get_execution_role(sagemaker_session=sagemaker_session)

suffix_user = getpass.getuser() if getpass.getuser() != "root" else "user"
endpoint_name = f"anthrofold-endpoint-{suffix_user}-{int(time.time()) % 100000}"
customer_bucket = f"sagemaker-{S3_BUCKET_REGION}-{account_id}"
output_prefix = "anthrofold/output/"

print(f"SageMaker region: {SAGEMAKER_REGION}")
print(f"S3 bucket region: {S3_BUCKET_REGION}")
print(f"Account: {account_id}")
print(f"Role: {role}")
print(f"Endpoint: {endpoint_name}")
print(f"Instance: {INSTANCE_TYPE}")
print(f"Bucket: {customer_bucket}")

## S3 bucket

Async inference uses S3 to hand request payloads in and prediction outputs back. By default we use the standard SageMaker bucket for your account (`sagemaker-<region>-<account-id>`); it's created on the fly if it doesn't already exist.

In [ ]:
try:
    s3.head_bucket(Bucket=customer_bucket)
    print(f"Using existing bucket: {customer_bucket}")
except s3.exceptions.ClientError:
    if S3_BUCKET_REGION == "us-east-1":
        s3.create_bucket(Bucket=customer_bucket)
    else:
        s3.create_bucket(
            Bucket=customer_bucket,
            CreateBucketConfiguration={"LocationConstraint": S3_BUCKET_REGION},
        )
    print(f"Created bucket: {customer_bucket}")

## Deploy and wait for readiness

This cell starts the endpoint, waits for SageMaker to mark it `InService`, then waits until the model finishes syncing its databases before declaring the endpoint ready. SageMaker sometimes flips `InService` before the model is actually ready to serve, so the second check matters.

When everything is ready, the endpoint name is written to `endpoint_name.txt` so the next notebooks can pick it up automatically. Expect about an hour on first deploy.

In [ ]:
from botocore.exceptions import ClientError, WaiterError

model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

try:
    model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=endpoint_name,
        async_inference_config=AsyncInferenceConfig(
            output_path=f"s3://{customer_bucket}/{output_prefix}",
            max_concurrent_invocations_per_instance=1,
        ),
        container_startup_health_check_timeout=3600,
        model_data_download_timeout=600,
        wait=False,
    )
    print(f"Endpoint {endpoint_name} is deploying.")
except ClientError as exc:
    print(f"Could not start deployment: {exc}")
    raise

try:
    sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
    print(f"Endpoint is InService: {endpoint_name}")
except WaiterError as exc:
    desc = sm.describe_endpoint(EndpointName=endpoint_name)
    print(f"Endpoint status: {desc.get('EndpointStatus')}")
    print(f"Failure reason: {desc.get('FailureReason', '')}")

    cleanup_errors = []

    endpoint_config_name = desc.get("EndpointConfigName")
    model_name = None
    if endpoint_config_name:
        try:
            config_desc = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
            variants = config_desc.get("ProductionVariants") or []
            model_name = variants[0].get("ModelName") if variants else None
        except Exception as cleanup_exc:
            print(f"Could not inspect endpoint config for cleanup: {cleanup_exc}")

    try:
        sm.delete_endpoint(EndpointName=endpoint_name)
        print(f"Deleted failed endpoint: {endpoint_name}")
    except Exception as cleanup_exc:
        cleanup_errors.append(f"endpoint {endpoint_name}: {cleanup_exc}")
        print(f"Endpoint delete skipped or failed: {cleanup_exc}")

    for _ in range(120):
        try:
            sm.describe_endpoint(EndpointName=endpoint_name)
            print("Endpoint still deleting; waiting 30s")
            time.sleep(30)
        except Exception:
            print("Endpoint is gone")
            break

    if endpoint_config_name:
        try:
            sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
            print(f"Deleted endpoint config: {endpoint_config_name}")
        except Exception as cleanup_exc:
            cleanup_errors.append(f"endpoint-config {endpoint_config_name}: {cleanup_exc}")
            print(f"Endpoint config delete skipped or failed: {cleanup_exc}")

    if model_name:
        try:
            sm.delete_model(ModelName=model_name)
            print(f"Deleted model: {model_name}")
        except Exception as cleanup_exc:
            cleanup_errors.append(f"model {model_name}: {cleanup_exc}")
            print(f"Model delete skipped or failed: {cleanup_exc}")

    if cleanup_errors:
        print("\n*** WARNING: cleanup did not fully succeed; these resources may still be LIVE and BILLING — delete them manually in the SageMaker console: ***")
        for _err in cleanup_errors:
            print(f"    - {_err}")
    print("Deployment failed. If this was an insufficient-capacity failure, re-run this cell to try again.")
    raise exc

log_group = f"/aws/sagemaker/Endpoints/{endpoint_name}"
timeout_seconds = 7200
poll_seconds = 60
start = time.time()
deadline = start + timeout_seconds
start_ms = int((start - timeout_seconds) * 1000)

print("Waiting for model runtime readiness.")
while time.time() < deadline:
    ready = logs.filter_log_events(
        logGroupName=log_group,
        filterPattern='"AnthroFold model ready"',
        startTime=start_ms,
        limit=1,
    ).get("events", [])
    if ready:
        print("Model runtime is ready.")
        break

    finished = logs.filter_log_events(
        logGroupName=log_group,
        filterPattern='"DB sync finished"',
        startTime=start_ms,
    ).get("events", [])
    if len(finished) >= 2:
        print("Model runtime is ready.")
        break

    elapsed_min = int((time.time() - start) / 60)
    print(f"[{elapsed_min} min] runtime initialization in progress")
    time.sleep(poll_seconds)
else:
    raise TimeoutError(f"Runtime readiness marker was not found within {timeout_seconds}s")

# Save the endpoint name to a sidecar file so the invoke and cleanup notebooks can pick it up automatically.
from pathlib import Path
sidecar = Path("endpoint_name.txt")
sidecar.write_text(endpoint_name + "\n")
print(f"Wrote endpoint name to {sidecar.resolve()}")

## You're all set

Print the endpoint details for reference. The invoke and cleanup notebooks will pick the endpoint name up automatically from `endpoint_name.txt`, so you usually don't need to copy anything by hand.

In [ ]:
print(f"ENDPOINT_NAME = {endpoint_name!r}")
print(f"CUSTOMER_BUCKET = {customer_bucket!r}")
print(f"SAGEMAKER_REGION = {SAGEMAKER_REGION!r}")